# Extract PDF Tables to Local Files

Extracts tabular data from a financial PDF (AT&T 2Q2025 Financial & Operational
Schedules) using **pdfplumber** and writes the result to local Parquet/CSV files.

This notebook was originally authored for Microsoft Fabric (OneLake/ABFSS paths,
`mssparkutils`, Spark Lakehouse writes). It has been **rewritten to run locally in
VS Code** with no cloud dependencies:

| Original (Fabric) | Local rewrite |
|---|---|
| Copy PDF from OneLake via `mssparkutils`/Spark `binaryFile` | Read PDF directly from `RawData/PDF/` |
| `pip install` at runtime | Fail fast with a clear install hint |
| `extract_tables()` → numbers crammed into one text cell | Column-aware text parser that splits each numeric column |
| Spark write to Lakehouse table | pandas → Parquet + CSV (Spark is unnecessary and unreliable on this box) |

**Pipeline:** configure paths → check deps → locate PDF → extract → normalise → write.


In [1]:
# --- Configuration -----------------------------------------------------------
from pathlib import Path

# Optional: pin a specific file in RawData/PDF. If None, the first *.pdf is used.
PDF_FILENAME: str | None = None

# Output table naming.
OUTPUT_PREFIX = "att_pdf_"


def find_project_root(start: Path) -> Path:
    """Walk up from `start` until a folder containing 'RawData' is found.

    Makes the notebook robust to VS Code's working directory (project root or the
    notebook's own folder).
    """
    for candidate in (start, *start.parents):
        if (candidate / "RawData").is_dir():
            return candidate
    raise FileNotFoundError(
        f"Could not locate project root (a parent of {start} containing 'RawData')."
    )


PROJECT_ROOT = find_project_root(Path.cwd())
PDF_DIR = PROJECT_ROOT / "RawData" / "PDF"
OUTPUT_DIR = PROJECT_ROOT / "RawData" / "scratch" / "extracted_pdf_tables"

print(f"Project root : {PROJECT_ROOT}")
print(f"PDF dir      : {PDF_DIR}")
print(f"Output dir   : {OUTPUT_DIR}")


Project root : C:\Users\PS\Documents\Python-Exp
PDF dir      : C:\Users\PS\Documents\Python-Exp\RawData\PDF
Output dir   : C:\Users\PS\Documents\Python-Exp\RawData\scratch\extracted_pdf_tables


In [2]:
# --- Dependencies ------------------------------------------------------------
# Fail fast with an actionable message instead of installing packages at runtime.
import importlib.util
import sys

_REQUIRED = {"pdfplumber": "pdfplumber", "pandas": "pandas", "pyarrow": "pyarrow"}
_missing = [pip_name for mod, pip_name in _REQUIRED.items()
            if importlib.util.find_spec(mod) is None]
if _missing:
    raise ModuleNotFoundError(
        "Missing required packages: " + ", ".join(_missing)
        + f"\nInstall them with:\n    {sys.executable} -m pip install "
        + " ".join(_missing)
    )

import re

import pandas as pd
import pdfplumber

print(f"python {sys.version.split()[0]} | pandas {pd.__version__} | "
      f"pdfplumber {pdfplumber.__version__}")


python 3.14.4 | pandas 3.0.3 | pdfplumber 0.11.10


In [3]:
# --- Locate and validate the PDF --------------------------------------------
def resolve_pdf(pdf_dir: Path, filename: str | None) -> Path:
    """Return the PDF to process, validating it exists and is non-empty."""
    if not pdf_dir.is_dir():
        raise FileNotFoundError(f"PDF directory does not exist: {pdf_dir}")

    if filename:
        path = pdf_dir / filename
        if not path.is_file():
            raise FileNotFoundError(f"Configured PDF not found: {path}")
    else:
        pdfs = sorted(pdf_dir.glob("*.pdf"))
        if not pdfs:
            raise FileNotFoundError(f"No '*.pdf' files found in {pdf_dir}")
        if len(pdfs) > 1:
            print(f"Multiple PDFs found; using the first: {[p.name for p in pdfs]}")
        path = pdfs[0]

    if path.stat().st_size == 0:
        raise ValueError(f"PDF is empty: {path}")
    return path


PDF_PATH = resolve_pdf(PDF_DIR, PDF_FILENAME)
print(f"Using PDF: {PDF_PATH.name}  ({PDF_PATH.stat().st_size / 1_048_576:.2f} MB)")


Using PDF: ATT_Fin-Op_Non_GAAP_Recon.pdf.pdf  (0.87 MB)


In [4]:
# --- Extraction helpers ------------------------------------------------------
# These AT&T schedules are laid out by text position, not ruled grid lines, so
# pdfplumber's default extract_tables() crams every number of a row into a single
# text cell. Instead we read each line of text and peel the right-aligned numeric
# columns off the end, leaving the line-item label on the left.

_NIL = "�"  # AT&T's "nil" em-dash arrives mis-decoded as the replacement char

# Layout lines that are neither data nor section titles.
_NOISE = re.compile(
    r"^(at&t inc\.?|financial data|dollars in millions.*|unaudited.*|"
    r"subscribers and connections.*|.*percent.*|[\d,]+\s+[\d,]+.*change.*|"
    r"\(000,000\)|consolidated supplementary data)\s*$",
    re.IGNORECASE,
)
# A standalone page number / stray footnote line.
_PAGE_FOOTER = re.compile(r"^\W*\d{1,3}\W*$")


def _tokenize(line: str) -> list[str]:
    """Split a line into tokens, gluing '$'+number and number+'%' back together."""
    parts = line.replace(_NIL, " — ").split()  # normalise nil to em-dash
    glued: list[str] = []
    i = 0
    while i < len(parts):
        if parts[i] == "$" and i + 1 < len(parts):
            glued.append("$ " + parts[i + 1])
            i += 2
        else:
            glued.append(parts[i])
            i += 1
    out: list[str] = []
    i = 0
    while i < len(glued):
        if i + 1 < len(glued) and glued[i + 1] == "%":
            out.append(glued[i] + " %")
            i += 2
        else:
            out.append(glued[i])
            i += 1
    return out


def _is_value(tok: str) -> bool:
    """True if a token is a number (with $, commas, %, parens) or a nil dash."""
    core = re.sub(r"[\$%,()\s—-]", "", tok)
    if core == "":
        return "—" in tok  # nil marker, optionally followed by %
    return bool(re.fullmatch(r"\d+(?:\.\d+)?", core))


def split_label_values(line: str) -> tuple[str, list[str]]:
    """Return (label, [values]) by peeling value tokens off the right end."""
    toks = _tokenize(line)
    k = len(toks)
    while k > 0 and _is_value(toks[k - 1]):
        k -= 1
    label = " ".join(toks[:k]).strip()
    values = [t.replace("—", "").strip() for t in toks[k:]]
    return label, values


def _page_title(lines: list[str]) -> str:
    """The first non-noise, value-free line on a page = the schedule title."""
    for line in lines:
        line = line.strip()
        if not line or _NOISE.match(line) or _PAGE_FOOTER.match(line):
            continue
        _, values = split_label_values(line)
        if not values:
            return line
    return ""


def extract_tables(pdf_path: Path) -> pd.DataFrame:
    """Extract line items from every page into one tidy long-format DataFrame.

    Columns: page, page_title, line_item, col_1 .. col_n (one per numeric column).
    """
    rows: list[dict] = []
    with pdfplumber.open(pdf_path) as pdf:
        for pnum, page in enumerate(pdf.pages, start=1):
            lines = (page.extract_text() or "").split("\n")
            title = _page_title(lines)
            for line in lines:
                line = line.strip()
                if not line or _NOISE.match(line) or _PAGE_FOOTER.match(line):
                    continue
                label, values = split_label_values(line)
                if not values or not label:
                    continue
                row = {"page": pnum, "page_title": title, "line_item": label}
                row.update({f"col_{i}": v for i, v in enumerate(values, start=1)})
                rows.append(row)

    if not rows:
        raise ValueError("No tabular rows were extracted from the PDF.")

    df = pd.DataFrame(rows)
    value_cols = sorted((c for c in df.columns if c.startswith("col_")),
                        key=lambda c: int(c.split("_")[1]))
    return df[["page", "page_title", "line_item", *value_cols]]


print("Extraction helpers defined.")


Extraction helpers defined.


In [5]:
# --- Run extraction ----------------------------------------------------------
tables_df = extract_tables(PDF_PATH)

print(f"Extracted {len(tables_df)} line items across "
      f"{tables_df['page'].nunique()} pages.\n")
print("Schedules found:")
for page, title in tables_df.groupby("page")["page_title"].first().items():
    print(f"  p{page:>2}: {title}")

tables_df.head(20)


Extracted 360 line items across 19 pages.

Schedules found:
  p 1: 2Q2025
  p 2: Consolidated Statements of Income
  p 3: Consolidated Balance Sheets
  p 4: Consolidated Statements of Cash Flows
  p 5: Supplementary Financial Data
  p 6: COMMUNICATIONS SEGMENT
  p 7: Mobility
  p 8: Business Wireline
  p 9: Consumer Wireline
  p10: LATIN AMERICA SEGMENT
  p11: SUPPLEMENTAL SEGMENT RECONCILIATION
  p12: SUPPLEMENTAL SEGMENT RECONCILIATION
  p13: Discussion and Reconciliation of Non-GAAP Measures
  p14: control. Because we do not control these entities, management excludes these results when evaluating the performance of our
  p15: Segment and Business Unit EBITDA, EBITDA Margin and EBITDA Service Margin
  p16: Adjusting Items
  p17: Adjusted Operating Income, Adjusted Operating Income Margin,
  p18: Net Debt to Adjusted EBITDA
  p19: Supplemental Operational Measures


,page,page_title,line_item,col_1,col_2,col_3,col_4,col_5,col_6,col_7,col_8,col_9
0,1,2Q2025,July,"23,",2025,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,Consolidated Statements of Income,Service,"$ 25,292","$ 25,006",1.1 %,"$ 50,430","$ 49,848",1.2 %,NaN,NaN,NaN
2,2,Consolidated Statements of Income,Equipment,"5,555","4,791",15.9 %,"11,043","9,977",10.7 %,NaN,NaN,NaN
3,2,Consolidated Statements of Income,Total Operating Revenues,"30,847","29,797",3.5 %,"61,473","59,825",2.8 %,NaN,NaN,NaN
4,2,Consolidated Statements of Income,Equipment,"5,738","4,815",19.2 %,"11,432","9,958",14.8 %,NaN,NaN,NaN
5,2,Consolidated Statements of Income,and amortization shown separately below),"6,412","6,627",(3.2) %,"12,751","13,438",(5.1) %,NaN,NaN,NaN
6,2,Consolidated Statements of Income,"Selling, general and administrative","6,945","7,043",(1.4) %,"14,090","14,064",0.2 %,NaN,NaN,NaN
7,2,Consolidated Statements of Income,Asset impairments and abandonments and restruc...,,480,%,504,639,(21.1) %,NaN,NaN,NaN
8,2,Consolidated Statements of Income,Depreciation and amortization,"5,251","5,072",3.5 %,"10,441","10,119",3.2 %,NaN,NaN,NaN
9,2,Consolidated Statements of Income,Total Operating Expenses,"24,346","24,037",1.3 %,"49,218","48,218",2.1 %,NaN,NaN,NaN


In [6]:
# --- Normalise money/percent columns to numeric ------------------------------
def to_number(value) -> float | None:
    """Convert a financial string ('$ 1,234', '(56)', '7.9 %') to a float.

    Parentheses denote negatives; nil/blank values become NA.
    """
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return pd.NA
    s = str(value).strip()
    if s in ("", "—"):
        return pd.NA
    negative = s.startswith("(") and s.endswith(")")
    s = re.sub(r"[\$,%()\s]", "", s)
    if s in ("", "-"):
        return pd.NA
    try:
        number = float(s)
    except ValueError:
        return pd.NA
    return -number if negative else number


VALUE_COLS = [c for c in tables_df.columns if c.startswith("col_")]

numeric_df = tables_df.copy()
for col in VALUE_COLS:
    numeric_df[col] = numeric_df[col].map(to_number).astype("Float64")

print(f"Normalised {len(VALUE_COLS)} value columns to numeric (Float64).")
numeric_df.head(20)


Normalised 9 value columns to numeric (Float64).


,page,page_title,line_item,col_1,col_2,col_3,col_4,col_5,col_6,col_7,col_8,col_9
0,1,2Q2025,July,23.0,2025.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1,2,Consolidated Statements of Income,Service,25292.0,25006.0,1.1,50430.0,49848.0,1.2,<NA>,<NA>,<NA>
2,2,Consolidated Statements of Income,Equipment,5555.0,4791.0,15.9,11043.0,9977.0,10.7,<NA>,<NA>,<NA>
3,2,Consolidated Statements of Income,Total Operating Revenues,30847.0,29797.0,3.5,61473.0,59825.0,2.8,<NA>,<NA>,<NA>
4,2,Consolidated Statements of Income,Equipment,5738.0,4815.0,19.2,11432.0,9958.0,14.8,<NA>,<NA>,<NA>
5,2,Consolidated Statements of Income,and amortization shown separately below),6412.0,6627.0,3.2,12751.0,13438.0,5.1,<NA>,<NA>,<NA>
6,2,Consolidated Statements of Income,"Selling, general and administrative",6945.0,7043.0,1.4,14090.0,14064.0,0.2,<NA>,<NA>,<NA>
7,2,Consolidated Statements of Income,Asset impairments and abandonments and restruc...,<NA>,480.0,<NA>,504.0,639.0,21.1,<NA>,<NA>,<NA>
8,2,Consolidated Statements of Income,Depreciation and amortization,5251.0,5072.0,3.5,10441.0,10119.0,3.2,<NA>,<NA>,<NA>
9,2,Consolidated Statements of Income,Total Operating Expenses,24346.0,24037.0,1.3,49218.0,48218.0,2.1,<NA>,<NA>,<NA>


In [7]:
# --- Write outputs -----------------------------------------------------------
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Combined tables: raw-string version (Parquet + CSV) and the numeric version.
raw_parquet = OUTPUT_DIR / f"{OUTPUT_PREFIX}tables.parquet"
raw_csv = OUTPUT_DIR / f"{OUTPUT_PREFIX}tables.csv"
numeric_parquet = OUTPUT_DIR / f"{OUTPUT_PREFIX}tables_numeric.parquet"

tables_df.to_parquet(raw_parquet, index=False)
tables_df.to_csv(raw_csv, index=False, encoding="utf-8-sig")
numeric_df.to_parquet(numeric_parquet, index=False)

# One CSV per schedule/page for convenient inspection.
by_page_dir = OUTPUT_DIR / "by_page"
by_page_dir.mkdir(exist_ok=True)
page_files: list[str] = []
for page, group in tables_df.groupby("page"):
    title = group["page_title"].iloc[0] or f"page_{page}"
    slug = re.sub(r"[^A-Za-z0-9]+", "_", str(title)).strip("_").lower()[:50]
    out_file = by_page_dir / f"p{page:02d}_{slug}.csv"
    group.dropna(axis=1, how="all").to_csv(out_file, index=False, encoding="utf-8-sig")
    page_files.append(out_file.name)

print("Wrote:")
for path in (raw_parquet, raw_csv, numeric_parquet):
    print(f"  {path}  ({path.stat().st_size:,} bytes)")
print(f"  {len(page_files)} per-page CSVs in {by_page_dir}")


Wrote:
  C:\Users\PS\Documents\Python-Exp\RawData\scratch\extracted_pdf_tables\att_pdf_tables.parquet  (21,262 bytes)
  C:\Users\PS\Documents\Python-Exp\RawData\scratch\extracted_pdf_tables\att_pdf_tables.csv  (36,631 bytes)
  C:\Users\PS\Documents\Python-Exp\RawData\scratch\extracted_pdf_tables\att_pdf_tables_numeric.parquet  (19,602 bytes)
  19 per-page CSVs in C:\Users\PS\Documents\Python-Exp\RawData\scratch\extracted_pdf_tables\by_page
